# INF8225 — Projet : MK-UNet (Équipe 5)
**Salem Christopher — 2144272 | Ezzedine Mahdi — 2146059**  
Chargé de labo : Aziz Attia — Hiver 2026

---
Reproduction de *MK-UNet: Multi-Kernel Lightweight CNN for Medical Image Segmentation*  
Rahman & Marculescu, ICCV 2025 CVAMD — [ArXiv](https://arxiv.org/abs/2509.18493)

### Plan d'action
| Étape | Section |
|-------|---------|
| Set Up | §1 |
| DataLoader | §2 |
| Architecture | §3 |
| Entraînement | §4 |
| Métriques | §5 |
| Visualisations | §6 |
| Analyse | §7 |


---
# 1. Set Up

In [ ]:
# ── Installation (décommenter si nécessaire) ──────────────────────────
# PyTorch AMD / ROCm → voir requirements.txt
# !pip install -r requirements.txt


In [ ]:
import os, sys, time, logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchinfo import summary
import wandb

sys.path.insert(0, os.path.abspath('.'))
from mkunet_network import MKUNet
from utils import build_loader, structure_loss, dice_score, iou_score
from utils import clip_gradient, RunningAverage, count_params_flops

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Configuration globale ─────────────────────────────────────────────
IMG_SIZE    = 352
BATCH_TRAIN = 8
BATCH_EVAL  = 4
EPOCHS      = 100          # papier : 200
LR          = 5e-4
GRAD_CLIP   = 0.5
DATA_ROOT   = Path('./data')
CKPT_DIR    = Path('./checkpoints')
RESULTS_DIR = Path('./results')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

WANDB_PROJECT = 'INF8225-MKUNet'

print("Configuration chargée ✓")


In [ ]:
# ── WandB login ───────────────────────────────────────────────────────
# wandb.login()   # décommenter au premier lancement


---
# 2. DataLoader

Structure canonique attendue pour chaque dataset :
```
data/<DATASET>/train/images/   train/masks/
               val/images/     val/masks/
               test/images/    test/masks/
```
Les cellules ci-dessous téléchargent et reorganisent chaque dataset
dans cette structure, puis vérifient visuellement quelques paires image/masque.


In [ ]:
# ── Helper : affichage d'une grille image / masque ────────────────────
def denorm(t, mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)):
    img = t.clone().cpu().float()
    for c,(m,s) in enumerate(zip(mean,std)):
        img[c] = img[c]*s + m
    return img.permute(1,2,0).numpy().clip(0,1)

def show_samples(loader, n=4, title=''):
    imgs, masks = next(iter(loader))
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    fig.suptitle(title, fontsize=12)
    for i in range(n):
        axes[0,i].imshow(denorm(imgs[i])); axes[0,i].axis('off')
        axes[1,i].imshow(masks[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[1,i].axis('off')
    axes[0,0].set_ylabel('Image'); axes[1,0].set_ylabel('Masque')
    plt.tight_layout(); plt.show()
    print(f"Images : {imgs.shape} | Masques : {masks.shape} | "
          f"Masque min/max : {masks.min()}/{masks.max()}")


## 2.1 BUSI — Breast Ultrasound Images Dataset

In [ ]:
# ── Téléchargement BUSI (Kaggle) ──────────────────────────────────────
# Nécessite un compte Kaggle et le fichier ~/.kaggle/kaggle.json
# !kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
# !unzip -q breast-ultrasound-images-dataset.zip -d data/BUSI_raw

# ── Réorganisation BUSI vers structure canonique ──────────────────────
# BUSI contient : Dataset_BUSI_with_GT/{benign,malignant}/{img.png, img_mask.png}
# On fusionne benign + malignant, on ignore 'normal' (pas de lésion).

import shutil, random
from sklearn.model_selection import train_test_split

def organize_busi(raw_dir='data/BUSI_raw/Dataset_BUSI_with_GT',
                  out_dir='data/BUSI', seed=42):
    random.seed(seed)
    pairs = []
    for cls in ('benign', 'malignant'):
        cls_dir = Path(raw_dir) / cls
        if not cls_dir.exists(): continue
        imgs = sorted([p for p in cls_dir.iterdir()
                       if '_mask' not in p.name and p.suffix == '.png'])
        for img_p in imgs:
            mask_p = img_p.with_name(img_p.stem + '_mask.png')
            if mask_p.exists():
                pairs.append((img_p, mask_p))

    tr, tmp = train_test_split(pairs, test_size=0.2, random_state=seed)
    val, te  = train_test_split(tmp,   test_size=0.5, random_state=seed)

    for split, data_pairs in [('train',tr),('val',val),('test',te)]:
        (Path(out_dir)/split/'images').mkdir(parents=True, exist_ok=True)
        (Path(out_dir)/split/'masks').mkdir(parents=True, exist_ok=True)
        for img_p, mask_p in data_pairs:
            shutil.copy(img_p,  Path(out_dir)/split/'images'/img_p.name)
            shutil.copy(mask_p, Path(out_dir)/split/'masks'/mask_p.name)
    print(f"BUSI : train={len(tr)}, val={len(val)}, test={len(te)}")

# organize_busi()   # décommenter une seule fois

# ── Vérification ──────────────────────────────────────────────────────
busi_train = build_loader('data/BUSI/train/images','data/BUSI/train/masks',
                           BATCH_TRAIN, IMG_SIZE, split='train', augment=True)
busi_val   = build_loader('data/BUSI/val/images',  'data/BUSI/val/masks',
                           BATCH_EVAL,  IMG_SIZE, split='test',  augment=False)
busi_test  = build_loader('data/BUSI/test/images', 'data/BUSI/test/masks',
                           BATCH_EVAL,  IMG_SIZE, split='test',  augment=False)
print(f"BUSI  — train: {len(busi_train.dataset)} | val: {len(busi_val.dataset)} | test: {len(busi_test.dataset)}")
show_samples(busi_train, title='BUSI — paires entraînement')


## 2.2 ClinicDB — Polyp Dataset

In [ ]:
# ── Téléchargement ClinicDB (Google Drive via gdown) ──────────────────
# import gdown
# gdown.download_folder('https://drive.google.com/drive/folders/1FPJr5f91uUCikxMvkwtZSEnYHemTZq1P',
#                        output='data/ClinicDB', quiet=False)

clinicdb_train = build_loader('data/ClinicDB/train/images','data/ClinicDB/train/masks',
                               BATCH_TRAIN, IMG_SIZE, split='train', augment=True)
clinicdb_val   = build_loader('data/ClinicDB/val/images',  'data/ClinicDB/val/masks',
                               BATCH_EVAL,  IMG_SIZE, split='test')
clinicdb_test  = build_loader('data/ClinicDB/test/images', 'data/ClinicDB/test/masks',
                               BATCH_EVAL,  IMG_SIZE, split='test')
print(f"ClinicDB — train: {len(clinicdb_train.dataset)} | val: {len(clinicdb_val.dataset)} | test: {len(clinicdb_test.dataset)}")
show_samples(clinicdb_train, title='ClinicDB — paires entraînement')


## 2.3 ColonDB — Polyp Dataset

In [ ]:
# ── Téléchargement ColonDB ────────────────────────────────────────────
# import gdown
# gdown.download_folder('https://drive.google.com/drive/folders/1u4_8dMztnEBUaX-w3XfUR3jXLBhpccPA',
#                        output='data/ColonDB', quiet=False)

colondb_train = build_loader('data/ColonDB/train/images','data/ColonDB/train/masks',
                              BATCH_TRAIN, IMG_SIZE, split='train', augment=True)
colondb_val   = build_loader('data/ColonDB/val/images',  'data/ColonDB/val/masks',
                              BATCH_EVAL,  IMG_SIZE, split='test')
colondb_test  = build_loader('data/ColonDB/test/images', 'data/ColonDB/test/masks',
                              BATCH_EVAL,  IMG_SIZE, split='test')
print(f"ColonDB — train: {len(colondb_train.dataset)} | val: {len(colondb_val.dataset)} | test: {len(colondb_test.dataset)}")
show_samples(colondb_train, title='ColonDB — paires entraînement')


## 2.4 ISIC18 — Skin Lesion Dataset

In [ ]:
# ── Téléchargement ISIC18 (Kaggle) ───────────────────────────────────
# !kaggle datasets download -d bhaveshmittal/melanoma-cancer-dataset
# Ou depuis le site officiel ISIC 2018 : https://challenge.isic-archive.com/data/#2018
#
# Réorganisation : les masques ISIC ont le suffixe _segmentation.png
# Les images et masques sont appariés sur le radical du nom de fichier.

def organize_isic18(img_raw='data/ISIC18_raw/ISIC2018_Task1-2_Training_Input',
                    mask_raw='data/ISIC18_raw/ISIC2018_Task1_Training_GroundTruth',
                    out_dir='data/ISIC18', seed=42):
    imgs  = sorted(Path(img_raw).glob('*.jpg'))
    pairs = []
    for img_p in imgs:
        mask_p = Path(mask_raw) / (img_p.stem + '_segmentation.png')
        if mask_p.exists():
            pairs.append((img_p, mask_p))

    tr, tmp = train_test_split(pairs, test_size=0.2, random_state=seed)
    val, te  = train_test_split(tmp,   test_size=0.5, random_state=seed)

    for split, data_pairs in [('train',tr),('val',val),('test',te)]:
        (Path(out_dir)/split/'images').mkdir(parents=True, exist_ok=True)
        (Path(out_dir)/split/'masks').mkdir(parents=True, exist_ok=True)
        for img_p, mask_p in data_pairs:
            shutil.copy(img_p,  Path(out_dir)/split/'images'/img_p.name)
            shutil.copy(mask_p, Path(out_dir)/split/'masks'/(img_p.stem+'_mask.png'))
    print(f"ISIC18 : train={len(tr)}, val={len(val)}, test={len(te)}")

# organize_isic18()  # décommenter une seule fois

isic_train = build_loader('data/ISIC18/train/images','data/ISIC18/train/masks',
                           BATCH_TRAIN, IMG_SIZE, split='train', augment=True)
isic_val   = build_loader('data/ISIC18/val/images',  'data/ISIC18/val/masks',
                           BATCH_EVAL,  IMG_SIZE, split='test')
isic_test  = build_loader('data/ISIC18/test/images', 'data/ISIC18/test/masks',
                           BATCH_EVAL,  IMG_SIZE, split='test')
print(f"ISIC18 — train: {len(isic_train.dataset)} | val: {len(isic_val.dataset)} | test: {len(isic_test.dataset)}")
show_samples(isic_train, title='ISIC18 — paires entraînement')


## 2.5 DSB18 — Data Science Bowl 2018 (Cell Nuclei)

In [ ]:
# ── Téléchargement DSB18 ─────────────────────────────────────────────
# !kaggle competitions download -c data-science-bowl-2018
#
# DSB18 : chaque image dans son propre dossier avec sous-dossier masks/
# On fusionne les masques individuels en un seul masque binaire.

def organize_dsb18(raw_dir='data/DSB18_raw/stage1_train',
                   out_dir='data/DSB18', seed=42):
    import cv2
    cases = [d for d in Path(raw_dir).iterdir() if d.is_dir()]
    pairs = []
    for case in cases:
        img_paths = list((case/'images').glob('*.png'))
        if not img_paths: continue
        img_p = img_paths[0]
        # Fusion de tous les masques individuels
        mask_dir = case / 'masks'
        combined = None
        for mp in mask_dir.glob('*.png'):
            m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            combined = m if combined is None else np.maximum(combined, m)
        if combined is not None:
            pairs.append((img_p, combined, case.name))

    tr, tmp = train_test_split(pairs, test_size=0.2, random_state=seed)
    val, te  = train_test_split(tmp,   test_size=0.5, random_state=seed)

    for split, data_pairs in [('train',tr),('val',val),('test',te)]:
        (Path(out_dir)/split/'images').mkdir(parents=True, exist_ok=True)
        (Path(out_dir)/split/'masks').mkdir(parents=True, exist_ok=True)
        for img_p, mask_arr, name in data_pairs:
            shutil.copy(img_p, Path(out_dir)/split/'images'/(name+'.png'))
            cv2.imwrite(str(Path(out_dir)/split/'masks'/(name+'.png')), mask_arr)
    print(f"DSB18 : train={len(tr)}, val={len(val)}, test={len(te)}")

# organize_dsb18()   # décommenter une seule fois

dsb_train = build_loader('data/DSB18/train/images','data/DSB18/train/masks',
                          BATCH_TRAIN, IMG_SIZE, split='train', augment=True)
dsb_val   = build_loader('data/DSB18/val/images',  'data/DSB18/val/masks',
                          BATCH_EVAL,  IMG_SIZE, split='test')
dsb_test  = build_loader('data/DSB18/test/images', 'data/DSB18/test/masks',
                          BATCH_EVAL,  IMG_SIZE, split='test')
print(f"DSB18  — train: {len(dsb_train.dataset)} | val: {len(dsb_val.dataset)} | test: {len(dsb_test.dataset)}")
show_samples(dsb_train, title='DSB18 — paires entraînement')


## 2.6 EM — Electron Microscopy (30 images)

In [ ]:
# ── EM : très petit dataset (30 images) ──────────────────────────────
# Source : ISBI 2012 Segmentation Challenge
# https://imagej.net/events/isbi-2012-segmentation-challenge
# Télécharger train-volume.tif et train-labels.tif
# On découpe les stacks tif en images individuelles.

def organize_em(vol_path='data/EM_raw/train-volume.tif',
                lbl_path='data/EM_raw/train-labels.tif',
                out_dir='data/EM', seed=42):
    from PIL import Image as PILImage
    vol = PILImage.open(vol_path)
    lbl = PILImage.open(lbl_path)
    n_frames = vol.n_frames
    pairs = list(range(n_frames))

    tr, tmp = train_test_split(pairs, test_size=0.4, random_state=seed)
    val, te  = train_test_split(tmp,   test_size=0.5, random_state=seed)

    for split, idxs in [('train',tr),('val',val),('test',te)]:
        (Path(out_dir)/split/'images').mkdir(parents=True, exist_ok=True)
        (Path(out_dir)/split/'masks').mkdir(parents=True, exist_ok=True)
        for idx in idxs:
            vol.seek(idx); vol.save(Path(out_dir)/split/'images'/f'em_{idx:03d}.png')
            lbl.seek(idx); lbl.save(Path(out_dir)/split/'masks' /f'em_{idx:03d}.png')
    print(f"EM : train={len(tr)}, val={len(val)}, test={len(te)}")

# organize_em()   # décommenter une seule fois

# EM est en grayscale → rgb=False
em_train = build_loader('data/EM/train/images','data/EM/train/masks',
                         min(4, BATCH_TRAIN), IMG_SIZE, split='train', augment=True, rgb=False)
em_val   = build_loader('data/EM/val/images',  'data/EM/val/masks',
                         2, IMG_SIZE, split='test', rgb=False)
em_test  = build_loader('data/EM/test/images', 'data/EM/test/masks',
                         2, IMG_SIZE, split='test', rgb=False)
print(f"EM     — train: {len(em_train.dataset)} | val: {len(em_val.dataset)} | test: {len(em_test.dataset)}")
show_samples(em_train, title='EM — paires entraînement')


---
# 3. Architecture — MKUNet

## Blocs principaux

**MKDC** (*Multi-Kernel Depthwise Convolution*) : P convolutions dépthwise en parallèle avec des noyaux différents (1×1, 3×3, 5×5). Chaque branche applique DWConv → BN → ReLU6. Les sorties sont sommées.

**MKIR** (*Multi-Kernel Inverted Residual*) : bloc MobileNetV2 où la DWConv standard est remplacée par MKDC. Schéma : PW-expand → MKDC → somme → PW-project + skip connection.

**CA** (*Channel Attention*) + **SA** (*Spatial Attention*) : modules CBAM appliqués séquentiellement avant chaque étage décodeur → **MKIRA** = CA → SA → MKIR.

**GAG** (*Grouped Attention Gate*) : modulation des skip connections. Calcule ψ = σ(W_ψ·ReLU(W_g·g + W_x·x)) et renvoie x·ψ pour supprimer les features encodeur non pertinentes.


In [ ]:
# ── Instanciation du modèle (variante de base : ~0.8 M params) ────────
model = MKUNet(
    num_classes      = 1,
    in_channels      = 3,
    channels         = [16, 32, 64, 96, 160],
    blocks_per_stage = [1, 1, 1, 1, 1],
    kernel_sizes     = [1, 3, 5],
    expansion_ratio  = 2,
    gag_kernel       = 3,
).to(DEVICE)

summary(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE),
        col_names=['output_size','num_params','trainable'], depth=3)


In [ ]:
# ── FLOPs & Paramètres ────────────────────────────────────────────────
flops_G, params_M = count_params_flops(model, IMG_SIZE)
print(f"FLOPs : {flops_G:.3f} G  |  Paramètres : {params_M:.3f} M")


---
# 4. Entraînement

## Fonction de perte : Structure Loss

Combinaison BCE + IoU avec pondération bord-de-masque (PraNet) :
$$w = 1 + 5|\text{AvgPool}(y) - y|$$
$$\mathcal{L} = w \cdot \text{BCE}(\hat{p}, y) + \left(1 - \frac{\sum \hat{p} \cdot y \cdot w + 1}{\sum (\hat{p}+y) \cdot w - \sum \hat{p} \cdot y \cdot w + 1}\right)$$

## Protocole
- Optimiseur : AdamW, lr = 5e-4, weight_decay = 1e-4
- Scheduler  : CosineAnnealingLR (T_max = EPOCHS, η_min = 1e-6)
- Multi-scale training : redimensionnement aléatoire ×0.75 / ×1 / ×1.25
- Sauvegarde du meilleur checkpoint (val Dice)


In [ ]:
# ── Fonctions d'évaluation et d'entraînement ──────────────────────────

@torch.no_grad()
def evaluate(model, loader):
    """Calcule mDice et mIoU sur un loader val/test (avec resize à la résolution originale)."""
    model.eval()
    total_dice = total_iou = n = 0
    for imgs, masks, orig_sizes, _ in loader:
        imgs  = imgs.to(DEVICE)
        masks = masks.float().to(DEVICE)
        preds = model(imgs)

        for i in range(len(imgs)):
            orig_h, orig_w = int(orig_sizes[0][i]), int(orig_sizes[1][i])
            p = F.interpolate(preds[i:i+1], size=(orig_h, orig_w),
                              mode='bilinear', align_corners=False)
            p = torch.sigmoid(p).squeeze()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)   # norm locale

            g = F.interpolate(masks[i:i+1], size=(orig_h, orig_w),
                              mode='nearest').squeeze()
            pb = (p >= 0.5).float(); gb = (g >= 0.2).float()
            total_dice += dice_score(pb, gb)
            total_iou  += iou_score(pb,  gb)
            n += 1
    return total_dice / n, total_iou / n


def train_one_epoch(model, loader, optimizer, size_rates=(0.75, 1.0, 1.25)):
    model.train()
    meter = RunningAverage()
    for imgs, masks in loader:
        for rate in size_rates:
            optimizer.zero_grad()
            x = imgs.to(DEVICE); y = masks.float().to(DEVICE)
            if rate != 1.0:
                ts = int(round(IMG_SIZE * rate / 32) * 32)
                x = F.interpolate(x, (ts, ts), mode='bilinear', align_corners=True)
                y = F.interpolate(y, (ts, ts), mode='nearest')
            loss = structure_loss(model(x), y)
            loss.backward()
            clip_gradient(optimizer, GRAD_CLIP)
            optimizer.step()
            if rate == 1.0:
                meter.update(loss.item(), imgs.size(0))
    return meter.avg


## 4.1 Choix du dataset d'entraînement

Modifie `ACTIVE_DATASET` pour changer de dataset.

In [ ]:
# ── Choix du dataset actif ────────────────────────────────────────────
# Options : 'BUSI' | 'ClinicDB' | 'ColonDB' | 'ISIC18' | 'DSB18' | 'EM'
ACTIVE_DATASET = 'ClinicDB'

loaders = {
    'BUSI'    : (busi_train,     busi_val,     busi_test),
    'ClinicDB': (clinicdb_train, clinicdb_val, clinicdb_test),
    'ColonDB' : (colondb_train,  colondb_val,  colondb_test),
    'ISIC18'  : (isic_train,     isic_val,     isic_test),
    'DSB18'   : (dsb_train,      dsb_val,      dsb_test),
    'EM'      : (em_train,       em_val,       em_test),
}
train_loader, val_loader, test_loader = loaders[ACTIVE_DATASET]
print(f"Dataset actif : {ACTIVE_DATASET}")
print(f"  train={len(train_loader.dataset)} | val={len(val_loader.dataset)} | test={len(test_loader.dataset)}")


In [ ]:
# ── Initialisation run ────────────────────────────────────────────────
model = MKUNet(num_classes=1, in_channels=3).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

RUN_ID   = f"{ACTIVE_DATASET}_MKUNet_{datetime.now().strftime('%m%d_%H%M')}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

wandb.init(project=WANDB_PROJECT, name=RUN_ID, config={
    'dataset': ACTIVE_DATASET, 'epochs': EPOCHS,
    'batch_size': BATCH_TRAIN, 'lr': LR,
    'img_size': IMG_SIZE, 'channels': [16,32,64,96,160],
    'kernel_sizes': [1,3,5], 'loss': 'structure_loss',
    'flops_G': flops_G, 'params_M': params_M,
})
print(f"Run : {RUN_ID}")


In [ ]:
# ── Boucle d'entraînement ─────────────────────────────────────────────
history = {'train_loss': [], 'val_dice': [], 'val_iou': []}
best_dice, best_epoch = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    t0        = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_dice, val_iou = evaluate(model, val_loader)
    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)

    wandb.log({'epoch': epoch, 'train/loss': train_loss,
               'val/dice': val_dice, 'val/iou': val_iou, 'lr': lr_now})

    if val_dice > best_dice:
        best_dice, best_epoch = val_dice, epoch
        torch.save(model.state_dict(), CKPT_DIR / f'{RUN_ID}_best.pth')
        wandb.run.summary['best_val_dice']  = best_dice
        wandb.run.summary['best_epoch']     = best_epoch

    print(f"[{epoch:3d}/{EPOCHS}] loss={train_loss:.4f}  val_dice={val_dice:.4f}"
          f"  val_iou={val_iou:.4f}  lr={lr_now:.2e}  ({time.time()-t0:.0f}s)"
          + (" ★" if epoch == best_epoch else ""))

print(f"\nMeilleur val Dice : {best_dice:.4f} (epoch {best_epoch})")
wandb.finish()


---
# 5. Métriques — Évaluation par dataset

Chaque cellule charge le meilleur checkpoint entraîné sur ce dataset
et évalue mDice + mIoU sur le split test.


In [ ]:
# ── Helper d'évaluation ───────────────────────────────────────────────
def eval_dataset(ckpt_path, test_loader, dataset_name):
    m = MKUNet(num_classes=1, in_channels=3).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    dice, iou = evaluate(m, test_loader)
    print(f"  {dataset_name:<10}  mDice = {dice:.4f}   mIoU = {iou:.4f}")
    return {'dataset': dataset_name, 'mDice': round(dice,4), 'mIoU': round(iou,4)}


## 5.1 BUSI

In [ ]:
# Remplace le chemin par le checkpoint entraîné sur BUSI
busi_res = eval_dataset('checkpoints/<BUSI_RUN_ID>_best.pth', busi_test, 'BUSI')


## 5.2 ClinicDB

In [ ]:
clinicdb_res = eval_dataset('checkpoints/<CLINICDB_RUN_ID>_best.pth', clinicdb_test, 'ClinicDB')


## 5.3 ColonDB

In [ ]:
colondb_res = eval_dataset('checkpoints/<COLONDB_RUN_ID>_best.pth', colondb_test, 'ColonDB')


## 5.4 ISIC18

In [ ]:
isic_res = eval_dataset('checkpoints/<ISIC18_RUN_ID>_best.pth', isic_test, 'ISIC18')


## 5.5 DSB18

In [ ]:
dsb_res = eval_dataset('checkpoints/<DSB18_RUN_ID>_best.pth', dsb_test, 'DSB18')


## 5.6 EM

In [ ]:
em_res = eval_dataset('checkpoints/<EM_RUN_ID>_best.pth', em_test, 'EM')


## 5.7 Tableau récapitulatif

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────
all_res = [busi_res, clinicdb_res, colondb_res, isic_res, dsb_res, em_res]
df = pd.DataFrame(all_res)
print("\n" + "="*42)
print("  RÉSULTATS — MKUNet (base, ~0.8M params)")
print("="*42)
print(df.to_string(index=False))
print("="*42)
df.to_excel(RESULTS_DIR / 'summary_all_datasets.xlsx', index=False)
print(f"Tableau sauvegardé → {RESULTS_DIR}/summary_all_datasets.xlsx")


---
# 6. Visualisations

## 6.1 Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Courbes — {RUN_ID}', fontsize=12)
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], color='steelblue')
axes[0].set(title='Train Loss', xlabel='Epoch', ylabel='Loss')
axes[0].axvline(best_epoch, color='r', ls='--', alpha=.7, label=f'Best (ep {best_epoch})')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(ep, history['val_dice'], color='darkorange')
axes[1].set(title='Val mDice', xlabel='Epoch', ylabel='Dice')
axes[1].axhline(best_dice, color='g', ls=':', alpha=.7, label=f'Best={best_dice:.4f}')
axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(ep, history['val_iou'], color='mediumseagreen')
axes[2].set(title='Val mIoU', xlabel='Epoch', ylabel='IoU')
axes[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / f'curves_{RUN_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


## 6.2 Prédictions qualitatives côte à côte

In [ ]:
def qualitative_grid(model, loader, n=6, title=''):
    model.eval()
    imgs, masks, _, _ = next(iter(loader))
    with torch.no_grad():
        preds = torch.sigmoid(model(imgs.to(DEVICE))).cpu()

    fig, axes = plt.subplots(n, 3, figsize=(10, 3.5*n))
    fig.suptitle(title, fontsize=12)
    for col, lbl in enumerate(['Image','Masque GT','Prédiction']):
        axes[0,col].set_title(lbl, fontweight='bold')

    for i in range(n):
        p_bin = (preds[i].squeeze() >= 0.5).float()
        g_bin = (masks[i].squeeze() >= 0.2).float()
        dice  = dice_score(p_bin, g_bin)
        axes[i,0].imshow(denorm(imgs[i]))
        axes[i,1].imshow(g_bin,   cmap='gray', vmin=0, vmax=1)
        axes[i,2].imshow(p_bin,   cmap='gray', vmin=0, vmax=1)
        axes[i,0].set_ylabel(f'Dice={dice:.3f}', fontsize=9)
        for j in range(3): axes[i,j].axis('off')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'qualitative_{title}.png', dpi=130, bbox_inches='tight')
    plt.show()

# Exemples pour le dataset actif
qualitative_grid(model, test_loader, n=6, title=f'{ACTIVE_DATASET}_test')


## 6.3 Bar chart comparatif multi-dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Performances MKUNet par dataset', fontsize=13)
colors = plt.cm.Set2.colors

for ax, metric in zip(axes, ['mDice', 'mIoU']):
    vals = [r[metric] for r in all_res]
    names = [r['dataset'] for r in all_res]
    bars = ax.bar(names, vals, color=colors[:len(vals)], edgecolor='k', width=0.6)
    ax.set(title=metric, ylabel=metric, ylim=(0, 1.05))
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison_all_datasets.png', dpi=150, bbox_inches='tight')
plt.show()


---
# 7. Analyse & Rédaction du rapport

## 7.1 Architecture

*[À compléter dans le rapport final — base ci-dessous.]*

MKUNet adopte une architecture encodeur-décodeur en U à 5 étages. L'innovation centrale est le bloc **MKIR** : un résiduel inversé (style MobileNetV2) dont la convolution dépthwise est remplacée par **MKDC**, qui applique en parallèle des DWConv de noyaux 1×1, 3×3 et 5×5. Cela permet de capter des contextes multi-résolutions (textures fines vs. structures larges) sans coût paramétrique supplémentaire. Dans le décodeur, chaque étage suit le schéma **MKIRA** : CA → SA → MKIR, suivi d'un **GAG** qui module le skip connection encodeur pour filtrer les régions non pertinentes avant fusion.

## 7.2 Protocole expérimental

*[À compléter : datasets utilisés, splits train/val/test, hyperparamètres, durée, matériel.]*

## 7.3 Résultats obtenus

*[À compléter avec les vrais chiffres une fois l'entraînement terminé.]*

## 7.4 Analyse critique

**Forces :**
- Architecture ultra-légère (~0.8 M paramètres) compétitive avec des modèles bien plus lourds
- Capture multi-résolution sans explosion du nombre de paramètres
- Bonne généralisation entre modalités (endoscopie, ultrasons, dermoscopie)

**Limites observées :**
- Performances potentiellement plus faibles sur les très petits datasets (EM : 30 images)
- L'absence de pré-entraînement ImageNet limite le transfert de bas niveau
- Les lésions de petite taille ou à bords diffus restent difficiles à segmenter

**Pistes d'amélioration hors scope du projet :**
- Supervision profonde (deep supervision avec les têtes intermédiaires du décodeur)
- Data augmentation plus agressive (MixUp, CutMix) sur les petits datasets
